# Part 1 – EDA & Dataset Splitting

**Branch:** `feature/eda-dataset-splitting`


# Indoor Scene Change Detection



## 1. Environment Setup

In [ ]:
!pip install ultralytics pandas numpy scipy scikit-learn matplotlib seaborn opencv-python -q

import os, re, glob, json, shutil
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image as PILImage
from IPython.display import Image as IPyImage, display
from google.colab import files
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

print("Libraries ready.")


## 2. Upload Your Demo Reference/Query image Pair


In [ ]:
def upload_single_image(label):
    """Prompt the user to pick one image file (REFERENCE or QUERY) and save it under /content."""
    print(f"Select the {label} image:")
    uploaded = files.upload()
    assert len(uploaded) == 1, f"Please select exactly one {label} image."
    fname = next(iter(uploaded))
    ext = os.path.splitext(fname)[1] or '.jpg'
    dst = f'/content/user_{label.lower()}{ext}'
    with open(dst, 'wb') as f:
        f.write(uploaded[fname])
    print(f"Saved -> {dst}\n")
    return dst

USER_REFERENCE_PATH = upload_single_image('REFERENCE')
USER_QUERY_PATH  = upload_single_image('QUERY')

demo_reference_raw = cv2.cvtColor(cv2.imread(USER_REFERENCE_PATH), cv2.COLOR_BGR2RGB)
demo_query_raw  = cv2.cvtColor(cv2.imread(USER_QUERY_PATH),  cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))
axes[0].imshow(demo_reference_raw); axes[0].axis('off'); axes[0].set_title('Demo pair -- Reference', fontsize=12)
axes[1].imshow(demo_query_raw);  axes[1].axis('off'); axes[1].set_title('Demo pair -- Query', fontsize=12)
plt.tight_layout()
plt.show()

## 2. Load Dataset

In [ ]:
if not os.path.exists('/content/data.zip'):
    print("Select data.zip from your computer:")
    uploaded = files.upload()
    assert 'data.zip' in uploaded, "Upload a file named exactly 'data.zip' (Label Studio YOLO export)."
else:
    print("/content/data.zip already present -- skipping upload.")

In [ ]:
!unzip -qo /content/data.zip -d /content/custom_data
!ls -la /content/custom_data

In [ ]:
required = ['images', 'labels', 'classes.txt']
missing = [r for r in required if not os.path.exists(f'/content/custom_data/{r}')]
if missing:
    raise FileNotFoundError(f"Missing {missing} in /content/custom_data -- check the zip's folder structure.")
print("Folder structure looks correct.")

## 3. Exploratory Data Analysis — Raw Images

In [ ]:
RAW_IMAGE_DIR = '/content/custom_data/images'
RAW_LABEL_DIR = '/content/custom_data/labels'
CLASSES_TXT   = '/content/custom_data/classes.txt'

raw_images = sorted(glob.glob(os.path.join(RAW_IMAGE_DIR, '*.jpg')) +
                     glob.glob(os.path.join(RAW_IMAGE_DIR, '*.png')))
raw_labels = sorted(glob.glob(os.path.join(RAW_LABEL_DIR, '*.txt')))

with open(CLASSES_TXT) as f:
    class_names = [l.strip() for l in f if l.strip()]

print(f"Images found : {len(raw_images)}")
print(f"Label files  : {len(raw_labels)}")
print(f"Classes ({len(class_names)}): {class_names}")

images_without_labels = [
    p for p in raw_images
    if not os.path.exists(os.path.join(RAW_LABEL_DIR, os.path.splitext(os.path.basename(p))[0] + '.txt'))
]
print(f"Images with no matching label file: {len(images_without_labels)}")

In [ ]:
sample_n = min(300, len(raw_images))
rng = np.random.default_rng(42)
sample_paths = rng.choice(raw_images, size=sample_n, replace=False)

dims, file_sizes = [], []
for p in sample_paths:
    with PILImage.open(p) as im:
        dims.append(im.size)  # (w, h)
    file_sizes.append(os.path.getsize(p) / 1024)  # KB

widths, heights = zip(*dims)
dim_df = pd.DataFrame({'width': widths, 'height': heights, 'file_kb': file_sizes})
dim_df['aspect_ratio'] = dim_df['width'] / dim_df['height']

print("Image size / file-size summary (sampled):")
display(dim_df.describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(dim_df['width'], bins=20, color='#4C72B0'); axes[0].set_title('Image Width (px)')
axes[1].hist(dim_df['height'], bins=20, color='#DD8452'); axes[1].set_title('Image Height (px)')
axes[2].hist(dim_df['aspect_ratio'], bins=20, color='#55A868'); axes[2].set_title('Aspect Ratio (w/h)')
plt.tight_layout()
plt.show()

In [ ]:
class_counts = {c: 0 for c in class_names}
boxes_per_image = []

for lp in raw_labels:
    with open(lp) as f:
        lines = [l.strip() for l in f if l.strip()]
    boxes_per_image.append(len(lines))
    for line in lines:
        cls_id = int(line.split()[0])
        if 0 <= cls_id < len(class_names):
            class_counts[class_names[cls_id]] += 1

class_count_df = pd.Series(class_counts).sort_values(ascending=False)
total_instances = class_count_df.sum()

report_table = pd.DataFrame({
    'class': class_count_df.index,
    'instances': class_count_df.values,
    'pct_of_total': (class_count_df.values / total_instances * 100).round(1)
})
print("Object instances per class (raw labels):")
display(report_table)

plot_order = class_count_df.sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(9, max(3.5, 0.5 * len(plot_order))))
bars = ax.barh(plot_order.index, plot_order.values, color='#4C72B0', edgecolor='black', linewidth=0.5)
ax.bar_label(bars, padding=4, fontsize=10)
ax.set_xlim(0, plot_order.max() * 1.15)
ax.set_xlabel('Number of Instances', fontsize=11)
ax.set_title('Object Instances per Class - Raw Dataset', fontsize=13, fontweight='bold')
ax.grid(axis='x', alpha=0.3)
for spine in ['top', 'right']:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig('/content/class_distribution_raw.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def image_brightness_contrast(path):
    img = cv2.imread(path)
    if img is None:
        return None, None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return gray.mean(), gray.std()

bright_vals, contrast_vals = [], []
for p in sample_paths:
    b, c = image_brightness_contrast(p)
    if b is not None:
        bright_vals.append(b); contrast_vals.append(c)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(bright_vals, bins=25, color='#4C72B0')
axes[0].set_title('Raw Image Brightness (mean pixel intensity)')
axes[1].hist(contrast_vals, bins=25, color='#DD8452')
axes[1].set_title('Raw Image Contrast (pixel std-dev)')
plt.tight_layout()
plt.show()

print(f"Mean brightness: {np.mean(bright_vals):.1f}  |  Mean contrast: {np.mean(contrast_vals):.1f}")

# Sample grid of raw images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.ravel(), rng.choice(raw_images, size=8, replace=False)):
    img = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.set_title(os.path.basename(p), fontsize=7); ax.axis('off')
plt.suptitle('Sample Raw Images (before CLAHE)')
plt.tight_layout()
plt.show()

## 4. Preprocessing — CLAHE

### 4a. Apply CLAHE (originals are backed up to `images_original/`)

In [ ]:
def equalize_clahe(img_bgr, clip_limit=2.5, tile_grid_size=(8, 8)):
    """BGR image -> BGR image, CLAHE applied to the lightness channel only (colour preserved)."""
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

CLAHE_CLIP_LIMIT = 2.5
CLAHE_TILE_GRID = (8, 8)
ORIG_BACKUP_DIR = '/content/custom_data/images_original'

def preprocess_dataset_histogram_eq(image_dir, clip_limit=CLAHE_CLIP_LIMIT,
                                     tile_grid_size=CLAHE_TILE_GRID, backup=True):
    """Equalize every image in image_dir in place. Originals are copied to <image_dir>_original first; if that backup already exists the step is skipped so CLAHE is never applied twice."""
    paths = glob.glob(os.path.join(image_dir, '*.jpg')) + glob.glob(os.path.join(image_dir, '*.png'))
    if not paths:
        print(f"No images found in {image_dir} -- nothing to equalize.")
        return

    if backup:
        backup_dir = image_dir.rstrip('/') + '_original'
        if not os.path.exists(backup_dir):
            os.makedirs(backup_dir)
            for p in paths:
                shutil.copy2(p, os.path.join(backup_dir, os.path.basename(p)))
            print(f"Backed up {len(paths)} original images to {backup_dir}")
        else:
            print(f"Backup dir {backup_dir} already exists -- skipping backup "
                  f"(assuming equalization already ran once; re-running would double-apply CLAHE).")
            return

    n_ok, n_fail = 0, 0
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            n_fail += 1
            continue
        eq = equalize_clahe(img, clip_limit=clip_limit, tile_grid_size=tile_grid_size)
        cv2.imwrite(p, eq)
        n_ok += 1
    print(f"Histogram-equalized {n_ok} images in place in {image_dir} ({n_fail} unreadable/skipped).")

preprocess_dataset_histogram_eq('/content/custom_data/images')

# CLAHE-preprocessed version of the single demo pair uploaded above (in-memory + saved to disk),
# reused by every gallery/section below instead of looping over several dataset pairs.
demo_reference_bgr = cv2.imread(USER_REFERENCE_PATH)
demo_query_bgr  = cv2.imread(USER_QUERY_PATH)
demo_reference_pre = equalize_clahe(demo_reference_bgr, CLAHE_CLIP_LIMIT, CLAHE_TILE_GRID)
demo_query_pre  = equalize_clahe(demo_query_bgr,  CLAHE_CLIP_LIMIT, CLAHE_TILE_GRID)

DEMO_REFERENCE_PRE_PATH = '/content/user_reference_clahe.jpg'
DEMO_QUERY_PRE_PATH  = '/content/user_query_clahe.jpg'
cv2.imwrite(DEMO_REFERENCE_PRE_PATH, demo_reference_pre)
cv2.imwrite(DEMO_QUERY_PRE_PATH, demo_query_pre)
print('CLAHE applied to the demo pair ->', DEMO_REFERENCE_PRE_PATH, DEMO_QUERY_PRE_PATH)

### 4b. EDA Query Preprocessing

In [ ]:
bright_after, contrast_after = [], []
for p in sample_paths:
    b, c = image_brightness_contrast(p)   # sample_paths now point at the equalized files
    if b is not None:
        bright_after.append(b); contrast_after.append(c)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(bright_vals, bins=25, alpha=0.5, label='Before CLAHE', color='#4C72B0')
axes[0].hist(bright_after, bins=25, alpha=0.5, label='After CLAHE', color='#C44E52')
axes[0].set_title('Brightness: Before vs After'); axes[0].legend()

axes[1].hist(contrast_vals, bins=25, alpha=0.5, label='Before CLAHE', color='#4C72B0')
axes[1].hist(contrast_after, bins=25, alpha=0.5, label='After CLAHE', color='#C44E52')
axes[1].set_title('Contrast: Before vs After'); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Mean brightness  -> before CLAHE: {np.mean(bright_vals):.1f}, after CLAHE: {np.mean(bright_after):.1f}")
print(f"Mean contrast    -> before CLAHE: {np.mean(contrast_vals):.1f}, after CLAHE: {np.mean(contrast_after):.1f}")

In [ ]:
demo_reference_pre_rgb = cv2.cvtColor(demo_reference_pre, cv2.COLOR_BGR2RGB)
demo_query_pre_rgb  = cv2.cvtColor(demo_query_pre,  cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
axes[0, 0].imshow(demo_reference_raw);     axes[0, 0].set_title('Before CLAHE (demo -- Reference image)', fontsize=9); axes[0, 0].axis('off')
axes[0, 1].imshow(demo_reference_pre_rgb); axes[0, 1].set_title('After CLAHE (demo -- Reference image)', fontsize=9);  axes[0, 1].axis('off')
axes[1, 0].imshow(demo_query_raw);      axes[1, 0].set_title('Before CLAHE (demo -- Query image)', fontsize=9);  axes[1, 0].axis('off')
axes[1, 1].imshow(demo_query_pre_rgb);  axes[1, 1].set_title('After CLAHE (demo -- Query image)', fontsize=9);   axes[1, 1].axis('off')
plt.suptitle('Before vs After CLAHE -- Demo Pair')
plt.tight_layout()
plt.show()

## 5. Pair-Level Train / Validation / Test Split (70 / 15 / 15)

In [ ]:
IMAGE_DIR = '/content/custom_data/images'
LABEL_DIR = '/content/custom_data/labels'

ALL_IMAGES = glob.glob(os.path.join(IMAGE_DIR, '*.jpg')) + glob.glob(os.path.join(IMAGE_DIR, '*.png'))
print(f"Found {len(ALL_IMAGES)} images in {IMAGE_DIR}")

CHANGE_TYPES = ['add', 'delete', 'move']
KNOWN_ROOMS = ['bedroom', 'kitchen', 'dining', 'corridor', 'drawing']

FILENAME_PATTERN = re.compile(
    r'(?P<pair_id>\d+)-(?P<stage>before|after)-(?P<change_type>' + '|'.join(CHANGE_TYPES) + r')-'
    r'(?P<object>.+?)-(?P<room>' + '|'.join(KNOWN_ROOMS) + r')-cam-'
    r'(?P<camera_id>[^-]+)-(?P<member_roll>[^-]+)-(?P<seq>\d+)',
    re.IGNORECASE
)

def parse_filename(path):
    stem = os.path.splitext(os.path.basename(path))[0]
    m = FILENAME_PATTERN.search(stem)
    if not m:
        return None
    d = m.groupdict()
    d['path'] = path
    d['stage'] = d['stage'].lower()
    d['change_type'] = d['change_type'].lower()
    return d

parsed = [p for p in (parse_filename(f) for f in ALL_IMAGES) if p is not None]
skipped_files = [f for f in ALL_IMAGES if parse_filename(f) is None]
if skipped_files:
    print(f"NOTE: {len(skipped_files)} filenames didn't match add/delete/move and were dropped")
    print("(this includes any open/close/on/off pairs, plus any genuinely malformed names).")
    print("First 5 examples:")
    for f in skipped_files[:5]:
        print(" ", os.path.basename(f))

meta_df = pd.DataFrame(parsed)
print(f"\nKept {len(meta_df)} images with change_type in {CHANGE_TYPES}.")
meta_df.head()

In [ ]:
_pairs = []
_bad_pairs = []
for pid, grp in meta_df.groupby('pair_id'):
    references = grp[grp['stage'] == 'before']
    queries = grp[grp['stage'] == 'after']
    if len(references) != 1 or len(queries) != 1:
        _bad_pairs.append((pid, len(references), len(queries)))
        continue
    b, a = references.iloc[0], queries.iloc[0]
    _pairs.append({
        'pair_id': pid, 'reference_path': b['path'], 'query_path': a['path'],
        'change_type': a['change_type'], 'object': a['object'], 'room': a['room'],
    })
if _bad_pairs:
    print(f"WARNING: {len(_bad_pairs)} pair_ids didn't have exactly 1 reference + 1 query file -- skipped:")
    for pid, nb_, na_ in _bad_pairs[:10]:
        print(f"  pair_id={pid}: {nb_} reference, {na_} query")

ALL_PAIRS_DF = pd.DataFrame(_pairs)
print(f"Built {len(ALL_PAIRS_DF)} total reference/query pairs (Add/Delete/Move only).")

TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
assert abs(TRAIN_FRAC + VAL_FRAC + TEST_FRAC - 1.0) < 1e-9

_labels = ALL_PAIRS_DF['change_type']
_min_class_count = _labels.value_counts().min()
_stratify_ok = _min_class_count >= 2

train_ids, temp_ids = train_test_split(
    ALL_PAIRS_DF['pair_id'], train_size=TRAIN_FRAC, random_state=42,
    stratify=_labels if _stratify_ok else None)

temp_labels = ALL_PAIRS_DF.set_index('pair_id').loc[temp_ids, 'change_type']
_temp_stratify_ok = temp_labels.value_counts().min() >= 2
val_ids, test_ids = train_test_split(
    temp_ids, train_size=VAL_FRAC / (VAL_FRAC + TEST_FRAC), random_state=42,
    stratify=temp_labels if _temp_stratify_ok else None)

split_map = {pid: 'train' for pid in train_ids}
split_map.update({pid: 'validation' for pid in val_ids})
split_map.update({pid: 'test' for pid in test_ids})
ALL_PAIRS_DF['split'] = ALL_PAIRS_DF['pair_id'].map(split_map)

print("\nSplit sizes (pairs):")
print(ALL_PAIRS_DF['split'].value_counts())
print("\nChange-type balance across splits:")
print(ALL_PAIRS_DF.groupby('split')['change_type'].value_counts().unstack(fill_value=0))

In [ ]:
DATA_ROOT = '/content/data'
shutil.rmtree(DATA_ROOT, ignore_errors=True)   # start clean so a re-run never mixes old and new splits
missing_labels = []

def _copy_one(img_path, split):
    img_name = os.path.basename(img_path)
    stem = os.path.splitext(img_name)[0]
    dst_img_dir = os.path.join(DATA_ROOT, split, 'images')
    dst_lbl_dir = os.path.join(DATA_ROOT, split, 'labels')
    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_lbl_dir, exist_ok=True)
    shutil.copy2(img_path, os.path.join(dst_img_dir, img_name))
    label_src = os.path.join(LABEL_DIR, stem + '.txt')
    if os.path.exists(label_src):
        shutil.copy2(label_src, os.path.join(dst_lbl_dir, stem + '.txt'))
    else:
        missing_labels.append(img_name)

for _, row in ALL_PAIRS_DF.iterrows():
    _copy_one(row['reference_path'], row['split'])
    _copy_one(row['query_path'], row['split'])

if missing_labels:
    print(f"WARNING: {len(missing_labels)} images had no matching .txt label file -- "
          f"first 5: {missing_labels[:5]}")

print("Images copied per split:")
for split in ['train', 'validation', 'test']:
    n_img = len(glob.glob(f'{DATA_ROOT}/{split}/images/*'))
    n_lbl = len(glob.glob(f'{DATA_ROOT}/{split}/labels/*'))
    print(f"  {split}: {n_img} images, {n_lbl} labels")
    if n_img == 0:
        raise RuntimeError(f"{split} folder is empty -- check the split/copy logic above.")

### 5b. Dataset distribution charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

split_counts = ALL_PAIRS_DF['split'].value_counts().reindex(['train', 'validation', 'test'])
axes[0].bar(split_counts.index, split_counts.values, color=['#4C72B0', '#DD8452', '#55A868'])
for i, v in enumerate(split_counts.values):
    axes[0].text(i, v + 0.5, str(int(v)), ha='center', fontweight='bold')
axes[0].set_title('Pair-Level Train / Validation / Test Split')
axes[0].set_ylabel('Number of reference/query pairs')
axes[0].grid(alpha=0.3, axis='y')

type_counts = ALL_PAIRS_DF['change_type'].value_counts()
colors = plt.cm.Set2(np.linspace(0, 1, len(type_counts)))
axes[1].pie(type_counts.values, labels=[c.capitalize() for c in type_counts.index],
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[1].set_title('Change-Type Distribution (Add / Delete / Move)')

plt.tight_layout()
plt.savefig('/content/dataset_distribution.png', dpi=150)
plt.show()

by_split = ALL_PAIRS_DF.groupby('split')['change_type'].value_counts().unstack(fill_value=0)
by_split = by_split.reindex(['train', 'validation', 'test'])
by_split.plot(kind='bar', stacked=True, figsize=(8, 5), colormap='Set2')
plt.title('Change-Type Balance by Split')
plt.ylabel('Number of pairs')
plt.xticks(rotation=0)
plt.legend(title='change_type')
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/dataset_distribution_by_split.png', dpi=150)
plt.show()

## 6. Create data.yaml

In [ ]:
def create_data_yaml(path_to_classes_txt, path_to_data_yaml):
    if not os.path.exists(path_to_classes_txt):
        raise FileNotFoundError(f'classes.txt not found at {path_to_classes_txt}')
    with open(path_to_classes_txt, 'r') as f:
        classes = [line.strip() for line in f.readlines() if line.strip()]
    data = {
        'path': '/content/data',
        'train': 'train/images',
        'val': 'validation/images',
        'test': 'test/images',
        'nc': len(classes),
        'names': classes
    }
    with open(path_to_data_yaml, 'w') as f:
        yaml.dump(data, f, sort_keys=False)
    print(f"Created {path_to_data_yaml} with {len(classes)} classes")
    return classes

CLASSES = create_data_yaml('/content/custom_data/classes.txt', '/content/data.yaml')
!cat /content/data.yaml